In [1]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages

# ステートの定義
class State(TypedDict):
    # データを保存する属性
    messages: Annotated[list, add_messages]

# ステートグラフの作成
graph_builder = StateGraph(State)

In [3]:
# 言語モデルの定義
llm = ChatOpenAI(model_name=MODEL_NAME)

# チャットボットノードの作成
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

# グラフにチャットボットノードを追加
graph_builder.add_node("chatbot", chatbot)

# 開始ノードの指定
graph_builder.set_entry_point("chatbot")
# 終了ノードの指定
graph_builder.set_finish_point("chatbot")

# 実行可能なステートグラフの作成
graph = graph_builder.compile()

In [4]:
# グラフの実行
response = graph.invoke({"messages": [("user", "光の三原色は？")]})

# 結果の表示
print(response)

{'messages': [HumanMessage(content='光の三原色は？', additional_kwargs={}, response_metadata={}, id='d39b5d78-efbd-4813-b9d2-a729ae0c33d3'), AIMessage(content='光の三原色は、赤（Red）、緑（Green）、青（Blue）です。これらの色を組み合わせることで、さまざまな色を作り出すことができます。この原理は、加法混色と呼ばれ、特にディスプレイや照明の技術において広く使用されています。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 14, 'total_tokens': 95, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1e1f8f553d', 'finish_reason': 'stop', 'logprobs': None}, id='run-218ab9ea-43e4-48e9-9efb-822193a34294-0', usage_metadata={'input_tokens': 14, 'output_tokens': 81, 'total_tokens': 95, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]}


In [5]:
# 言語モデルからの回答のみを表示
print(response["messages"][-1].content)

光の三原色は、赤（Red）、緑（Green）、青（Blue）です。これらの色を組み合わせることで、さまざまな色を作り出すことができます。この原理は、加法混色と呼ばれ、特にディスプレイや照明の技術において広く使用されています。


In [ ]:
# グラフの実行と結果の表示
# 過去の会話は保持していない
def stream_graph_updates(user_input: str):
    # 結果をストリーミングで得る
    events = graph.stream({"messages": [("user", user_input)]})
    for event in events:
        for value in event.values():
            print("回答:", value["messages"][-1].content, flush=True)

# チャットボットのループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    print("質問:", user_input, flush=True)
    stream_graph_updates(user_input)

質問: おはよう
回答: おはようございます！今日はどんなことをお手伝いできますか？
質問: 眠い
回答: 眠い時は、しっかり休むことが大切です。少し仮眠を取ったり、リラックスする時間を設けたりするといいかもしれません。無理をせず、自分の体に優しくしてあげてくださいね。
質問: 日本の首都は？
回答: 日本の首都は東京です。
ありがとうございました!


In [8]:
# 記憶をもたせる
from langgraph.checkpoint.memory import MemorySaver

# チェックポインタの作成
memory = MemorySaver()

# 記憶を持つ実行可能なステートグラフの作成
memory_graph = graph_builder.compile(checkpointer=memory)

In [9]:
# グラフの実行と結果の表示
def stream_graph_updates(user_input: str):
    events = memory_graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        print(event["messages"][-1].content, flush=True)

# チャットボットのループ
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(user_input)

aから始まる単語を5つ教えて
もちろんです！「a」から始まる単語を5つ挙げます。

1. アップル (apple)
2. アート (art)
3. アニメ (anime)
4. アクション (action)
5. アドベンチャー (adventure)

他にも知りたい単語があれば教えてください！
3つめの単語ってどういう語源がある？
「アニメ（anime）」という単語の語源について説明します。

「アニメ」は、英語の「animation」から派生した言葉で、日本のアニメーションを指します。元々、「animation」はラテン語の「anima」から来ており、「魂」や「生命」を意味します。この語根は、物に命を吹き込むという意味合いから、動く画像を創り出すことに関連しています。

日本では、特に日本独自のスタイルのアニメーションを指す際に「アニメ」という言葉が使われるようになりました。日本のアニメは、1980年代以降、国内外で非常に人気を博し、文化の一部として広まりました。

このように、「アニメ」という言葉は、元々の語源から派生し、特定の文化的コンテキストを持つようになったものです。
ありがとうございました!
